In [9]:
import numpy as np
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import re
import seaborn as sns
import time

In [10]:
# read the wars csv file
wars = pd.read_csv('wars.csv')

#sort the wars by start date
wars = wars.sort_values('start')

# remove the columns "event", "category", "heritage_designation", "rank"
wars = wars.drop(columns = ['event', 'category', 'heritage_designation', 'rank'])

#change the name of columns
wars = wars.rename(columns = {'start':'date'})
wars = wars.rename(columns = {'eventLabel':'event'})

# change the date column in the format "YYYY"
wars['date'] = pd.to_datetime(wars['date']).dt.strftime('%Y')

#save file in a csv file
wars.to_csv('wars_cleaned.csv', index = False)

In [11]:
# Load the cleaned wars data
wars_cleaned = pd.read_csv('wars_cleaned.csv')

def get_word_count(url, retries=3, delay=5):
    for attempt in range(retries):
        response = requests.get(url)
        response.raise_for_status()  # Raise an HTTPError for bad responses
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract the main content of the page
        content = soup.find('div', {'id': 'mw-content-text'})

        if content:
            # Extract text and remove citations
            text = re.sub(r'\[\d+\]', '', content.get_text())
            
            # Count the number of words
            words = text.split()
            return len(words)
        else:
            return None

# Create a dataset with only events with more than 5000 words
events = []
for index, row in wars_cleaned.iterrows():
    url = row['articleLink']
    date = row['date']  # Assuming the date column is named 'date'
    words_count = get_word_count(url)
    if words_count and words_count >= 10000:
        events.append({
            'label': row['event'],  # Assuming the label column is named 'label'
            'url': url,
            'word_count': words_count,
            'date': date
        })
        print(f'Processed URL: {url} with word count: {words_count}')
    time.sleep(1)  # Add a delay between processing each URL to avoid rate limiting

# Save the events in a CSV file
relevant_wars = pd.DataFrame(events)



Processed URL: https://en.wikipedia.org/wiki/Sino-French_War with word count: 12973
Processed URL: https://en.wikipedia.org/wiki/First_Sino-Japanese_War with word count: 18797
Processed URL: https://en.wikipedia.org/wiki/First_Sino-Japanese_War with word count: 18797
Processed URL: https://en.wikipedia.org/wiki/Philippine_Revolution with word count: 15035
Processed URL: https://en.wikipedia.org/wiki/Philippine_Revolution with word count: 15035
Processed URL: https://en.wikipedia.org/wiki/Spanish%E2%80%93American_War with word count: 28499
Processed URL: https://en.wikipedia.org/wiki/Spanish%E2%80%93American_War with word count: 28499
Processed URL: https://en.wikipedia.org/wiki/Philippine%E2%80%93American_War with word count: 17836
Processed URL: https://en.wikipedia.org/wiki/Second_Boer_War with word count: 27067
Processed URL: https://en.wikipedia.org/wiki/Russo-Japanese_War with word count: 23055
Processed URL: https://en.wikipedia.org/wiki/Russian_Revolution_of_1905 with word count

In [12]:
#remove the columns "url" and "word_count"
relevant_wars = relevant_wars.drop(columns = ['url', 'word_count'])

#remove one of the rows that has same event name and same date
relevant_wars = relevant_wars.drop_duplicates()

#save the file in a csv file
relevant_wars.to_csv('relevant_wars.csv', index = False)

In [13]:
# Load the relevant_wars data if not already loaded
# relevant_wars = pd.read_csv('relevant_wars.csv')
relevant_wars = pd.read_csv('relevant_wars.csv')

# Group by 'label' and find the min and max dates for each group
grouped = relevant_wars.groupby('label')['date'].agg(['min', 'max']).reset_index()
grouped.columns = ['label', 'start', 'end']

# Merge the grouped data back with the original DataFrame
relevant_wars = pd.merge(relevant_wars, grouped, on='label')

# Check if 'start' and 'end' columns exist after the merge
if 'start' in relevant_wars.columns and 'end' in relevant_wars.columns:
    # Remove duplicates and keep only the rows with the 'start' and 'end' dates
    relevant_wars = relevant_wars.drop_duplicates(subset=['label', 'start', 'end'])
else:
    print("Error: 'start' and 'end' columns not found after merge.")

#remove date column
relevant_wars = relevant_wars.drop(columns = ['date'])

# Save the updated DataFrame to a CSV file
relevant_wars.to_csv('relevant_wars_with_dates.csv', index=False)

In [ ]:
demonyms = {
    'polish': 'Poland',
    'ukrainian': 'Ukraine',
    'american': 'United States',
    'british': 'United Kingdom',
    'french': 'France',
    'german': 'Germany',
    'italian': 'Italy',
    'italo': 'Italy',
    'japanese': 'Japan',
    'chinese': 'China',
    'russian': 'Russia',
    'hungarian': 'Hungary',
    'mexican': 'Mexico',
    'brazilian': 'Brazil',
    'canadian': 'Canada',
    'australian': 'Australia',
    'indian': 'India',
    'spanish': 'Spain',
    'portuguese': 'Portugal',
    'syrian': 'Syria',
    'egyptian': 'Egypt',
    'turkish': 'Turkey',
    'greek': 'Greece',
    'swedish': 'Sweden',
    'norwegian': 'Norway',
    'danish': 'Denmark',
    'finnish': 'Finland',
    'greco': 'Greece',  
    'irish': 'Ireland',
    'dutch': 'Netherlands',
    'belgian': 'Belgium',
    'swiss': 'Switzerland',
    'austrian': 'Austria',
    'czech': 'Czech Republic',
    'slovak': 'Slovakia',
    'palestine': 'Palestine',
    'arab': 'Israel',
    'iraqi': 'Iraq',
    'iranian': 'Iran',
    'afghan': 'Afghanistan',
    'pakistani': 'Pakistan',
    'indonesian': 'Indonesia',
    'malaysian': 'Malaysia',
    'vietnamese': 'Vietnam',
    'south korean': 'South Korea',
    'north korean': 'North Korea',
    'korean': 'South Korea',
    'korean': 'North Korea',
    'thai': 'Thailand',
    'philippine': 'Philippines',
    'balkan': 'Bulgaria',
    'balkan': 'Serbia',
    'balkan': 'Greece',
    'balkan': 'Romania',
    'balkan': 'Montenegro',
    'laotian': 'Laos',
    'rhodesian': 'Zimbabwe',
    'Lebanese': 'Lebanon',
    'salvadoran': 'El Salvador',
    'iran': 'Iran',
    'russo': 'Russia',
    'japanese': 'Japan',
    'croat': 'Croatia',
    'bosniak': 'Bosnia and Herzegovina',
    'abkhazia': 'Abkhazia',
    'chechen': 'Chechnya',
    'chechen': 'Russia',
    'kosovo': 'Kosovo',
    'kargil': 'India',
    'kargil': 'Pakistan',
    'xinhai': 'China',
    'world' : 'World',
    'october': 'Russia',
    'paulista': 'Brazil',
    'chaco': 'Paraguay',
    'chaco': 'Bolivia',
    'winter': 'Finland',
    'winter': ' Soviet Union',
    'indo': 'India',
    'kashmir':'India, Pakistan, China',
    'malayan': 'Malaysia',
    'suez': 'Egypt',
    'suez': 'United Kingdom',
    'suez': 'France',
    'suez': 'Israel',
    'cold': 'World',
    'cultural': 'China',
    'naxalite' : 'India',
    'six-day': 'Egypt, Syria, Jordan, Iraq',
    'six-day' : 'Israel',
    'attrition': 'Egypt, Soviet Union, Palestine, Syria, Cuba',
    'attrition': 'Israel',
    'white': 'Jordan',
    'white': 'Palestine, Syria',
    'kippur': 'Egypt, Syria',
    'kippur': 'Israel',
    'lebanese': 'Lebanon',
    'ogden':'Ethiopia, Cuba, South Yemen, Soviet Union',
    'ogden':'Somalia',
    'tanzanian': 'Tanzania',
    'sino': 'China',
    'falklands': 'United Kingdom',
    'falklands': 'Argentina',
    'nagorno': 'Armenia, Caucasus',
    'nagorno': 'Azerbaijan, Soviet Union',
    'gulf': 'Iraq',
    'gulf': 'Kuwait, Egypt, Saudi Arabia, France, United States, United Kingdom',
    'kargil': 'India',
    'kargil': 'Pakistan',
}

In [23]:
import pycountry
import pandas as pd
import re

# Load the relevant_wars data if not already loaded
relevant_wars = pd.read_csv('relevant_wars_with_dates.csv')

# Create a list of country names, including alternative names and official names
country_names = [country.name for country in pycountry.countries]
country_names += [country.official_name for country in pycountry.countries if hasattr(country, 'official_name')]

# Create a dictionary of demonyms to country names
demonyms = {
    'polish': ['Poland'],
    'ukrainian': ['Ukraine'],
    'american': ['United States'],
    'british': ['United Kingdom'],
    'french': ['France'],
    'german': ['Germany'],
    'italian': ['Italy'],
    'italo': ['Italy'],
    'japanese': ['Japan'],
    'chinese': ['China'],
    'russian': ['Russia'],
    'hungarian': ['Hungary'],
    'mexican': ['Mexico'],
    'brazilian': ['Brazil'],
    'canadian': ['Canada'],
    'australian': ['Australia'],
    'indian': ['India'],
    'spanish': ['Spain'],
    'portuguese': ['Portugal'],
    'syrian': ['Syria'],
    'egyptian': ['Egypt'],
    'turkish': ['Turkey'],
    'greek': ['Greece'],
    'swedish': ['Sweden'],
    'norwegian': ['Norway'],
    'danish': ['Denmark'],
    'finnish': ['Finland'],
    'anglo': ['United Kingdom'],
    'greco': ['Greece'],
    'irish': ['Ireland'],
    'dutch': ['Netherlands'],
    'belgian': ['Belgium'],
    'swiss': ['Switzerland'],
    'austrian': ['Austria'],
    'czech': ['Czech Republic'],
    'slovak': ['Slovakia'],
    'palestine': ['Palestine'],
    'arab': ['Israel'],
    'iraqi': ['Iraq'],
    'iranian': ['Iran'],
    'afghan': ['Afghanistan'],
    'pakistani': ['Pakistan'],
    'indonesian': ['Indonesia'],
    'malaysian': ['Malaysia'],
    'vietnamese': ['Vietnam'],
    'south korean': ['South Korea'],
    'north korean': ['North Korea'],
    'korean': ['South Korea', 'North Korea'],
    'thai': ['Thailand'],
    'philippine': ['Philippines'],
    'balkan': ['Bulgaria', 'Serbia', 'Greece', 'Romania', 'Montenegro'],
    'laotian': ['Laos'],
    'rhodesian': ['Zimbabwe'],
    'lebanese': ['Lebanon'],
    'salvadoran': ['El Salvador'],
    'tanzania': ['Tanzania'],
    'iran': ['Iran'],
    'russo': ['Russia'],
    'croat': ['Croatia'],
    'bosniak': ['Bosnia and Herzegovina'],
    'abkhazia': ['Abkhazia'],
    'chechen': ['Chechnya', 'Russia'],
    'kosovo': ['Kosovo'],
    'kargil': ['India', 'Pakistan'],
    'xinhai': ['China'],
    'world': ['World'],
    'october': ['Russia'],
    'paulista': ['Brazil'],
    'chaco': ['Paraguay', 'Bolivia'],
    'winter': ['Finland', 'Soviet Union'],
    'indo': ['India'],
    'kashmir': ['India', 'Pakistan', 'China'],
    'malayan': ['Malaysia'],
    'suez': ['Egypt', 'United Kingdom', 'France', 'Israel'],
    'cold': ['World'],
    'cultural': ['China'],
    'naxalite': ['India'],
    'six': ['Egypt', 'Syria', 'Jordan', 'Iraq', 'Israel'],
    'attrition': ['Egypt', 'Soviet Union', 'Palestine', 'Syria', 'Cuba', 'Israel'],
    'white': ['Jordan', 'Palestine', 'Syria'],
    'kippur': ['Egypt', 'Syria', 'Israel'],
    'ogden': ['Ethiopia', 'Cuba', 'South Yemen', 'Soviet Union', 'Somalia'],
    'tanzanian': ['Tanzania'],
    'sino': ['China'],
    'falklands': ['United Kingdom', 'Argentina'],
    'nagorno': ['Armenia', 'Caucasus', 'Azerbaijan', 'Soviet Union'],
    'gulf': ['Iraq', 'Kuwait', 'Egypt', 'Saudi Arabia', 'France', 'United States', 'United Kingdom'],
    'boer': ['South Africa', 'United Kingdom'],
    'soviet': ['Soviet Union'],
    'continuation': ['Soviet Union', 'Germany', 'Finland'],
    'pacific': ['United States', 'Japan', 'China', 'United Kingdom'],
    'august': ['China', 'Vietnam', 'Japan'],
    'bangladesh liberation': ['Bangladesh', 'India', 'Pakistan'],

}

# Function to preprocess event labels
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)  # Remove non-alphanumeric characters
    return text

# Function to get the best matching countries
def get_countries(event):
    event = preprocess_text(event)
    countries = set()
    
    # Check for exact matches with country names
    for country in country_names:
        if country.lower() in event:
            countries.add(country)
    
    # Check for matches with demonyms
    for demonym, country_list in demonyms.items():
        if demonym in event:
            countries.update(country_list)
    
    return sorted(countries)

# Apply the get_countries function to the 'label' column
relevant_wars['countries'] = relevant_wars['label'].apply(get_countries)

# Split the 'countries' column into multiple columns
countries_df = relevant_wars['countries'].apply(pd.Series)
countries_df.columns = [f'country{i+1}' for i in range(countries_df.shape[1])]

# Concatenate the original DataFrame with the new country columns
relevant_wars = pd.concat([relevant_wars, countries_df], axis=1)

# Drop the 'countries' column
relevant_wars = relevant_wars.drop(columns=['countries'])

# Display the first 30 rows of the updated DataFrame
print(relevant_wars.head(30))

# Save the updated DataFrame to a CSV file
relevant_wars.to_csv('relevant_wars_with_countries.csv', index=False)



                             label  start   end      country1        country2  \
0                  Sino-French War   1884  1884         China          France   
1          First Sino-Japanese War   1894  1895         China           Japan   
2            Philippine Revolution   1896  1896   Philippines             NaN   
3             Spanish–American War   1898  1898         Spain   United States   
4          Philippine–American War   1899  1899   Philippines   United States   
5                  Second Boer War   1899  1899  South Africa  United Kingdom   
6               Russo-Japanese War   1904  1905         Japan          Russia   
7       Russian Revolution of 1905   1905  1905        Russia             NaN   
8               Mexican Revolution   1910  1920        Mexico             NaN   
9                Xinhai Revolution   1911  1911         China             NaN   
10                First Balkan War   1912  1913      Bulgaria          Greece   
11               Second Balk